# Automatic training-window selection (one dataset per run)

Runs `run_window_selection.py` from the thesis repository: the framework is replayed four times
on the same stream (static, 60-day window, all history, and **automatic selection**, where the
promotion gate keeps the better of the two windows at every retrain).

**Run it three times, once per dataset.** For each run:
1. Set `DATASET` below to `"ieee"`, `"sparkov"` or `"baf"`.
2. **Add Input** for that dataset:
   - `ieee`: the competition **IEEE-CIS Fraud Detection** (`ieee-fraud-detection`). You must first open the
     competition page, click **Join Competition** / accept the rules, or the data cannot be attached.
   - `sparkov`: dataset **Credit Card Transactions Fraud Detection Dataset** by `kartik2112` (`fraud-detection`).
   - `baf`: dataset **Bank Account Fraud Dataset Suite (NeurIPS 2022)** by `sgpjesus`.
3. Settings: Accelerator **None**, Internet **On**.
4. **Save Version → Save & Run All (Commit)**. Output: `window_selection_<dataset>.zip`.

Expected time on Kaggle's CPU: BAF under 1 hour, IEEE-CIS about 2 hours, Sparkov about 3-5 hours.

In [ ]:
# ---- settings ----
DATASET = "ieee"     # "ieee", "sparkov" or "baf"
RUN_TESTS = True     # quick test suite first (about 3 min)
REPO_URL = "https://github.com/nishan469/fraud-mlops-thesis.git"

In [ ]:
import glob, os, shutil, subprocess, sys
WORK = "/kaggle/working"
CODE = f"{WORK}/fraud-mlops-thesis"
if not os.path.exists(f"{CODE}/run_window_selection.py"):
    if os.path.exists(CODE):
        shutil.rmtree(CODE)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE)
!pip install -q "lightgbm>=4.7" "mlflow>=3.16" pyarrow fastapi httpx pytest
import pandas, lightgbm, mlflow
print("pandas", pandas.__version__, "| lightgbm", lightgbm.__version__, "| mlflow", mlflow.__version__,
      "| CPUs", os.cpu_count())

In [ ]:
if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-W", "ignore", "tests"],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert r.returncode == 0, "tests failed: fix packages before the long run"

In [ ]:
# find the attached data
MARKER = {"ieee": "train_transaction.csv", "sparkov": "fraudTrain.csv", "baf": "Base.csv"}[DATASET]
hits = glob.glob(f"/kaggle/input/**/{MARKER}", recursive=True)
assert hits, f"{MARKER} not found: add the {DATASET} data as an input (see the top of the notebook)"
IN_DIR = os.path.dirname(hits[0])
print("Data:", IN_DIR, sorted(os.listdir(IN_DIR))[:8])

In [ ]:
# run the four replays; progress appears here
env = {**os.environ, "PYTHONUNBUFFERED": "1", "MLFLOW_DISABLE_AGENT_HINT": "1", "GIT_PYTHON_REFRESH": "quiet"}
p = subprocess.Popen([sys.executable, "-W", "ignore", "run_window_selection.py", "--dataset", DATASET,
                      "--in_dir", IN_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in p.stdout:
    if not any(s in line for s in ("INFO mlflow", "Registered model", "Created version", "Successfully registered")):
        print(line, end="")
assert p.wait() == 0, "run failed (see the lines above)"

In [ ]:
# results
import pandas as pd
OUT = f"outputs/{DATASET}/window_selection"
print(open(f"{OUT}/comparison.md").read())
display(pd.read_csv(f"{OUT}/comparison.csv"))

In [ ]:
# one zip to download (Output tab); the saved model files (mlruns) are left out
import zipfile
dest = f"{WORK}/window_selection_{DATASET}.zip"
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT):
        dirs[:] = [d for d in dirs if d != "mlruns"]
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "outputs"))
print(f"{os.path.getsize(dest) / 1e6:.1f} MB -> {dest}")